# Sales Data Overview

Load the sales dataset and summarize its structure, row count, key columns, and headline metrics.

In [1]:
import pandas as pd
from pathlib import Path
data_path = Path("data/sales_data.csv")
df = pd.read_csv(data_path)
df.head()

,date,product_category,region,units_sold,revenue_eur,cost_eur,channel
0,2025-01-15,Electronics,North,120,24000,18000,Online
1,2025-01-15,Electronics,South,85,17000,12750,Retail
2,2025-01-15,Furniture,North,45,13500,9450,Online
3,2025-01-15,Furniture,South,62,18600,13020,Retail
4,2025-01-15,Software,North,200,40000,8000,Online


In [5]:
print("Rows, columns:", df.shape)
print("Column names:", list(df.columns))
print("Data types:\n", df.dtypes)
print("Summary statistics:\n", df.describe(include="all"))

Rows, columns: (36, 7)
Column names: ['date', 'product_category', 'region', 'units_sold', 'revenue_eur', 'cost_eur', 'channel']
Data types:
 date                  str
product_category      str
region                str
units_sold          int64
revenue_eur         int64
cost_eur            int64
channel               str
dtype: object
Summary statistics:
               date product_category region  units_sold   revenue_eur  \
count           36               36     36   36.000000     36.000000   
unique           6                3      2         NaN           NaN   
top     2025-01-15      Electronics  North         NaN           NaN   
freq             6               12     18         NaN           NaN   
mean           NaN              NaN    NaN  130.333333  27980.555556   
std            NaN              NaN    NaN   78.090424  14035.217326   
min            NaN              NaN    NaN    5.000000   1000.000000   
25%            NaN              NaN    NaN   66.500000  17450.0000

## Headline Numbers

- Total rows and columns in the dataset
- Key dimensions: date, product_category, region, units_sold, revenue_eur, cost_eur, channel
- Quick business totals for revenue and units sold


In [3]:
print("Unique regions:", df["region"].nunique())
print("Unique product categories:", df["product_category"].nunique())
print("Total units sold:", df["units_sold"].sum())
print("Total revenue (EUR):", df["revenue_eur"].sum())
print("Total cost (EUR):", df["cost_eur"].sum())
print("Date range:", df["date"].min(), "to", df["date"].max())

Unique regions: 2
Unique product categories: 3
Total units sold: 4692
Total revenue (EUR): 1007300
Total cost (EUR): 451440
Date range: 2025-01-15 to 2025-06-15


## Anomaly Detection

Flag sales records where units sold is unusually low or changes sharply from month to month.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("data/sales_data.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df["month"] = df["date"].dt.to_period("M")

avg = df.groupby(["product_category","region"])["units_sold"].mean().reset_index(name="avg_units")
merged = df.merge(avg, on=["product_category","region"])
merged["pct_deviation"] = (merged["units_sold"] - merged["avg_units"]).abs() / merged["avg_units"]
deviation_alerts = merged[merged["pct_deviation"] > 0.5].copy()

monthly = (
    df.groupby(["product_category","region","month"])["units_sold"].sum().reset_index()
    .sort_values(["product_category","region","month"])
)
monthly["prev_units"] = monthly.groupby(["product_category","region"])["units_sold"].shift(1)
monthly["mom_change"] = (monthly["units_sold"] - monthly["prev_units"]).abs() / monthly["prev_units"]
mom_alerts = monthly[monthly["prev_units"].notna() & (monthly["mom_change"] > 0.4)].copy()

print("Deviation anomalies:
")
print(deviation_alerts[["date","product_category","region","units_sold","avg_units","pct_deviation"]].to_string(index=False))
print("
Month-over-month anomalies:
")
print(mom_alerts[["product_category","region","month","prev_units","units_sold","mom_change"]].to_string(index=False))

### Business Alert

- **Severity:** High
- **Issue:** May 2025, Electronics sales in the South region dropped to only 5 units, which is 94% below the category-region average of 79 units.
- **Signal:** This record also represents a 94% month-over-month decline from April (88 units) and is followed by a strong rebound to 110 units in June.
- **Possible explanations:** inventory or fulfillment disruption, promotional rollback, reporting/data entry error, or channel issue in South electronics.

**Recommendation:** Investigate the South Electronics workflow for May immediately; confirm whether this is a real sales collapse or a data/reporting problem, then take corrective action to avoid a repeat.


## Executive Summary

### Key Findings
- Software is the fastest-growing product category, with revenue rising 51% from January to June.
- The sales data contains 36 records across 3 product categories and 2 regions, totaling 4,692 units and 1,007,300 EUR in revenue.
- A high-severity anomaly was detected in May: Electronics sales in the South region collapsed to only 5 units.

### Anomaly Flags
- **South / Electronics / May 2025:** 5 units sold, 94% below the category-region average and an abnormal 94% month-over-month decline.
- **Follow-up signal:** sales rebounded to 110 units in June, suggesting the May drop was an outlier or operational disruption.

### Recommendations
- Investigate the South Electronics sales process in May immediately to determine whether this was a real demand issue or a reporting/data error.
- Maintain investment in Software, which shows the strongest growth momentum and should be prioritized for marketing and channel support.
- Add a monthly sales health check for category-region pairs to detect similar drops before they become larger business risks.
